# Human vs Machine Translation Detection Codebase

This Jupyter Notebook provides utilities for processing translation datasets in the context of detecting human versus machine-generated translations. It includes functions to load a pre-trained tokenizer, count tokens in text, and chunk CSV files containing translation data.

## Key Components:
- **Tokenizer Loading**: Load a saved tokenizer from a fine-tuned model directory.
- **Token Counting**: Count tokens in text strings using the tokenizer, excluding special tokens.
- **CSV Chunking**: Process a CSV file by merging consecutive rows within the same source language until the human translation column reaches a specified token limit. This creates larger chunks for analysis or training, while preserving metadata like original row IDs and token counts.

## Usage:
- Set the paths for the model directory, input CSV, and output CSV.
- Call the chunking function with customizable parameters (e.g., token limit, columns to exclude).
- The output is a new CSV with chunked data and a preview of the resulting DataFrame.

This notebook is designed for preprocessing data in machine translation detection tasks, ensuring efficient handling of variable-length texts.

In [ ]:
import os
import pandas as pd
from typing import List, Dict, Optional
from transformers import AutoTokenizer

In [ ]:
def load_saved_tokenizer(model_path: str):
    """
    Load tokenizer from a saved fine-tuned model directory.
    """
    tokenizer = AutoTokenizer.from_pretrained(model_path)
    return tokenizer

In [ ]:
def count_tokens(text: str, tokenizer) -> int:
    """
    Count tokens using the model tokenizer, without adding special tokens.
    """
    if text is None:
        text = ""
    text = str(text).strip()
    if not text:
        return 0

    token_ids = tokenizer.encode(text, add_special_tokens=False)
    return len(token_ids)

In [ ]:
def chunk_translation_csv_by_human_translation(
    input_csv_path: str,
    output_csv_path: str,
    tokenizer,
    token_limit: int = 300,
    language_column: str = "source language",
    anchor_column: str = "human translation",
    columns_to_exclude: Optional[List[str]] = None,
    join_str: str = " "
) -> pd.DataFrame:
    """
    Chunk rows by merging consecutive rows within the same source language until
    the anchor column (default: 'human translation') reaches or exceeds token_limit.

    Rules:
    - Only merge rows within the same source language
    - All columns except excluded ones are merged as text
    - Chunk boundary is decided by token count of anchor_column only
    """

    df = pd.read_csv(input_csv_path)

    if language_column not in df.columns:
        raise ValueError(f"Missing required language column: '{language_column}'")

    if anchor_column not in df.columns:
        raise ValueError(f"Missing required anchor column: '{anchor_column}'")

    if columns_to_exclude is None:
        columns_to_exclude = [language_column]

    # Preserve row order
    df = df.reset_index(drop=False).rename(columns={"index": "__original_row_id__"})

    # Columns that will be merged as text
    columns_to_merge = [c for c in df.columns if c not in columns_to_exclude]

    chunked_rows = []

    # Process each language separately
    for lang, group in df.groupby(language_column, sort=False):
        group = group.sort_values("__original_row_id__").reset_index(drop=True)

        current_chunk = {col: [] for col in columns_to_merge}
        current_row_ids = []
        current_token_count = 0
        chunk_id_within_lang = 0

        for _, row in group.iterrows():
            row_anchor_text = "" if pd.isna(row[anchor_column]) else str(row[anchor_column]).strip()
            row_anchor_tokens = count_tokens(row_anchor_text, tokenizer)

            # Add row contents into current chunk buffers
            for col in columns_to_merge:
                val = "" if pd.isna(row[col]) else str(row[col]).strip()
                if val:
                    current_chunk[col].append(val)

            current_row_ids.append(int(row["__original_row_id__"]))
            current_token_count += row_anchor_tokens

            # Finalize chunk once anchor reaches/passes token limit
            if current_token_count >= token_limit:
                merged_row = {}

                # keep language separate, not merged
                merged_row[language_column] = lang

                # merged text columns
                for col in columns_to_merge:
                    if col == "__original_row_id__":
                        continue
                    merged_row[col] = join_str.join(current_chunk[col]).strip()

                # metadata
                chunk_id_within_lang += 1
                merged_row["chunk_id"] = f"{lang}_{chunk_id_within_lang}"
                merged_row["num_rows_merged"] = len(current_row_ids)
                merged_row["original_row_ids"] = ",".join(map(str, current_row_ids))
                merged_row[f"{anchor_column}_token_count"] = current_token_count

                chunked_rows.append(merged_row)

                # reset
                current_chunk = {col: [] for col in columns_to_merge}
                current_row_ids = []
                current_token_count = 0

        # flush leftover chunk if non-empty
        if current_row_ids:
            merged_row = {}
            merged_row[language_column] = lang

            for col in columns_to_merge:
                if col == "__original_row_id__":
                    continue
                merged_row[col] = join_str.join(current_chunk[col]).strip()

            chunk_id_within_lang += 1
            merged_row["chunk_id"] = f"{lang}_{chunk_id_within_lang}"
            merged_row["num_rows_merged"] = len(current_row_ids)
            merged_row["original_row_ids"] = ",".join(map(str, current_row_ids))
            merged_row[f"{anchor_column}_token_count"] = current_token_count

            chunked_rows.append(merged_row)

    out_df = pd.DataFrame(chunked_rows)

    # Reorder columns a bit
    preferred_front = [
        "chunk_id",
        language_column,
        "num_rows_merged",
        "original_row_ids",
        f"{anchor_column}_token_count"
    ]
    ordered_cols = [c for c in preferred_front if c in out_df.columns] + [
        c for c in out_df.columns if c not in preferred_front
    ]
    out_df = out_df[ordered_cols]

    out_df.to_csv(output_csv_path, index=False, encoding="utf-8")
    print(f"Saved chunked CSV to: {output_csv_path}")
    print(f"Original rows: {len(df)}")
    print(f"Chunked rows: {len(out_df)}")

    return out_df

In [ ]:
final_model_path = "" # Set this to the path of the saved fine-tuned model directory that contains the tokenizer,

tokenizer = load_saved_tokenizer(final_model_path)
INPUT_CSV = "" # Set this to the path of the input CSV file that contains the translations to be chunked

OUTPUT_CSV = "" # Set this to the path where the chunked CSV file will be saved


chunked_df = chunk_translation_csv_by_human_translation(
    input_csv_path=INPUT_CSV,
    output_csv_path=OUTPUT_CSV,
    tokenizer=tokenizer,
    token_limit=300,
    language_column="source language",
    anchor_column="human translation",
    columns_to_exclude=["source language"],
    join_str=" "
)

chunked_df.head()